In [ ]:
import os
from getpass import getpass

# 공개용: 키를 코드에 하드코딩하지 않습니다.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")


In [ ]:
!pip install langchain_openai langgraph langchain_community

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
llm.invoke("야호")

In [ ]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

class State(TypedDict) :
  messages : Annotated[list[str], add_messages]

graph_builder = StateGraph(State)

In [ ]:
from langchain_core.tools import tool
import pytz
from datetime import datetime

@tool
def get_current_time(timezone : str, location : str) -> str :
  """현재 시간을 반환하는 함수"""
  try :
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    result = f"{timezone}({location}) 현재 시간{now}"
    return result
  except pytz.UnknownTimeZoneError :
    return f"알 수 없는 타임존 : {timezone}"

tools = [get_current_time]
tools[0].invoke({"timezone" : "Asia/Seoul", "location" : "서울"})

In [ ]:
my_agent = llm.bind_tools(tools)

In [ ]:
def generate(State : State) :
  return {"messages" : my_agent.invoke(State["messages"])}

graph_builder.add_node("generate", generate)

In [ ]:
import json
from langchain_core.messages import ToolMessage

class BasicToolNode :
  def __init__(self, tools : list) -> None :
    self.tools_by_name = {tool.name : tool for tool in tools}
  def __call__(self, inputs : dict) :
    if messages := inputs.get("messages", []) :
      message = messages[-1]
    else :
      raise ValueError("No message found in inputs")

    outputs = []

    for tool_call in message.tool_calls :
      tool_result = self.tools_by_name[tool_call["name"]].invoke(tool_call["args"])
      outputs.append(
          ToolMessage(content = json.dumps(tool_result),
                      name = tool_call["name"],
                      tool_call_id = tool_call["id"],)
      )
    return {"messages" : messages + outputs}

tool_node = BasicToolNode(tools = tools)
graph_builder.add_node("tools", tool_node)

In [ ]:
tmp = json.dumps({"a" : 1, "b" : 2})
display(tmp)
tmpdict = json.loads(tmp)
display(tmpdict)
display(tmpdict["a"])

In [ ]:
def route_tools(state : State) :
  if isinstance(state, list) :
    ai_message = state[-1]
  elif messages := state.get("messages", []) :
    ai_message = messages[-1]
  else :
    raise ValueError(f"tool_edge 입력 상태에서 메시지를 찾을 수 없습니다. : {state}")

  if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0 :
    return "tools"
  return END

graph_builder.add_edge(START, "generate")
graph_builder.add_conditional_edges(
    "generate", route_tools, {"tools" : "tools", END : END}
)
graph_builder.add_edge("tools", "generate")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
resp = graph.invoke({"messages" : ["지금 서울은 몇 시야?"]})
resp

In [ ]:
resp = graph.invoke({"messages" : ["런던은?"]})
display(resp)

In [ ]:
resp["messages"].append("5 더하기 3은?")
resp = graph.invoke(resp)
resp

In [ ]:
from langchain.tools import tool

@tool
def multiply(a : int, b : int) -> int :
  """Multiply two numbers"""
  return a*b

display(multiply.name)
display(multiply.description)
display(multiply.args)

In [ ]:
display(multiply.args_schema.model_json_schema())

In [ ]:
from pydantic import BaseModel, Field
class CalculatorInput(BaseModel) :
  a : int = Field(description = "first_number")
  b : int = Field(description = "second_number")

@tool(args_schema = CalculatorInput)
def multiply(a : int, b : int) -> int :
  """Multiply two numbers"""
  return a*b

display(multiply.name)
display(multiply.description)
display(multiply.args)

In [ ]:
@tool(parse_docstring = True)
def multiply(a:int, b:int) -> int:
  """Multiply two numbers.

  Args:
     a: The first number.
     b: The Second number.
  """
  return a*b

print(multiply.name)
print(multiply.description)
print(multiply.args)

In [ ]:
multiply.args_schema.model_json_schema()

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model='gpt-4o-mini',
    tools=tools,
)

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, AgentState
from langgraph.runtime import Runtime

llm = ChatOpenAI(model = "gpt-4o-mini")

BLOCKED_WORDS = ["바보", "멍청이", "나쁜말"]

@before_model
def content_filter_middleware(state : AgentState, runtime : Runtime) :
  """
  금지어를 필터링하는 미들웨어
  - 금지어 감지 시, 예외 발생으로 중단
  """

  if state["messages"] :
    last_msg = state["messages"][-1]
    content = getattr(last_msg, "content", str(last_msg))

    for word in  BLOCKED_WORDS :
      if word in content :
        print(f"[before model] 금지어 감지 : {word}")
        raise ValueError(f"부적절한 표현이 감지되었습니다. : '{word}'")
    print("[before model] 입력 검증 통과")
  return None

agent = create_agent(
    model = llm, tools = tools, middleware = [content_filter_middleware]
)

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
resp = agent.invoke({"messages" : "지금 런던은 몇 시야?"})
resp